# Korean Olympiad train/test deduplication

This Colab notebook removes likely benchmark leakage from the Korean Olympiad training pool.

- Training pool: rows where `source == 'olympiads'` from `ChuGyouk/AI-MO-NuminaMath-CoT-Ko`
- Held-out benchmark: `ChuGyouk/OlympiadBench-Math-Ko`
- Matching: normalized exact matching in English and Korean, fuzzy text matching, and multilingual semantic similarity
- Outputs: deduplicated Parquet dataset, candidate-match audit CSV, and JSON summary

Long-running outputs are checkpointed in Google Drive. After the first successful run, rerunning the notebook loads those checkpoints instead of downloading and processing all 859k source rows again.

In [ ]:
%pip install -q -U datasets huggingface_hub sentence-transformers rapidfuzz pandas pyarrow tqdm

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

print("Using HF_TOKEN from Colab Secrets" if HF_TOKEN else "HF_TOKEN not found; using public access")

In [ ]:
# ---- User controls ----
TRAIN_DATASET_ID = "ChuGyouk/AI-MO-NuminaMath-CoT-Ko"
TRAIN_SPLIT = "train"
TRAIN_SOURCE = "olympiads"
BENCHMARK_DATASET_ID = "ChuGyouk/OlympiadBench-Math-Ko"
BENCHMARK_SPLIT = "test"

DRIVE_ROOT = "/content/drive/MyDrive/Korean-TDCS/data/olympiad_dedup"
FORCE_REBUILD_FILTERED_DATA = False
FORCE_REBUILD_EMBEDDINGS = False

EMBEDDING_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_BATCH_SIZE = 4096  # Sized for a 96 GB RTX PRO 6000 Blackwell GPU.
REQUIRE_CUDA = True
USE_FP16_EMBEDDING_INFERENCE = True
SEMANTIC_TOP_K = 5
FUZZY_REMOVE_THRESHOLD = 97.0       # 0-100 RapidFuzz similarity
SEMANTIC_REMOVE_THRESHOLD = 0.985  # Deliberately strict to reduce false removals

# Audit and summary are downloaded automatically at the end. The Parquet file may be large.
DOWNLOAD_DEDUPLICATED_PARQUET = False

## 1. Paths and reusable helpers

In [ ]:
import hashlib
import json
import os
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

# Keep Hugging Face downloads in Drive as an additional cache across Colab sessions.
drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(drive_root / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(drive_root / "hf_cache" / "datasets")

FILTERED_TRAIN_PATH = drive_root / "numina_olympiads_train.parquet"
BENCHMARK_PATH = drive_root / "olympiadbench_math_ko_test.parquet"
TRAIN_EMBEDDINGS_PATH = drive_root / "numina_olympiads_english_embeddings.npy"
BENCHMARK_EMBEDDINGS_PATH = drive_root / "olympiadbench_english_embeddings.npy"
EMBEDDING_METADATA_PATH = drive_root / "embedding_metadata.json"
DEDUPLICATED_PATH = drive_root / "numina_olympiads_train_deduplicated.parquet"
MATCH_AUDIT_PATH = drive_root / "olympiad_duplicate_match_audit.csv"
SUMMARY_PATH = drive_root / "olympiad_deduplication_summary.json"
REMOVED_ROWS_PATH = drive_root / "olympiad_removed_training_rows.parquet"


def normalize_math_text(value):
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).lower()
    text = re.sub(r"\\(?:left|right|quad|qquad|,|!|;|:)", "", text)
    text = text.replace("$", "")
    text = re.sub(r"\s+", "", text)
    return text


def text_fingerprint(values):
    digest = hashlib.sha256()
    for value in values:
        digest.update(normalize_math_text(value).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()

## 2. Build or reload Drive checkpoints

The first run scans the full NuminaMath stream but retains only `source == 'olympiads'`. Subsequent runs load the saved Parquet file from Drive.

In [ ]:
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm

if FILTERED_TRAIN_PATH.exists() and not FORCE_REBUILD_FILTERED_DATA:
    print(f"Loading cached Olympiad training rows: {FILTERED_TRAIN_PATH}")
    train_dataset = load_dataset("parquet", data_files=str(FILTERED_TRAIN_PATH), split="train")
else:
    print("First run: streaming NuminaMath and retaining only source == 'olympiads'...")
    train_stream = load_dataset(
        TRAIN_DATASET_ID,
        split=TRAIN_SPLIT,
        streaming=True,
        token=HF_TOKEN,
    )
    retained_rows = []
    for row in tqdm(train_stream, desc="Scanning 859k NuminaMath rows"):
        if row.get("source") == TRAIN_SOURCE:
            retained_rows.append(row)
    train_dataset = Dataset.from_list(retained_rows)
    train_dataset.to_parquet(str(FILTERED_TRAIN_PATH))
    print(f"Saved filtered checkpoint: {FILTERED_TRAIN_PATH}")

if BENCHMARK_PATH.exists() and not FORCE_REBUILD_FILTERED_DATA:
    print(f"Loading cached benchmark: {BENCHMARK_PATH}")
    benchmark_dataset = load_dataset("parquet", data_files=str(BENCHMARK_PATH), split="train")
else:
    benchmark_dataset = load_dataset(
        BENCHMARK_DATASET_ID,
        split=BENCHMARK_SPLIT,
        token=HF_TOKEN,
    )
    benchmark_dataset.to_parquet(str(BENCHMARK_PATH))
    print(f"Saved benchmark checkpoint: {BENCHMARK_PATH}")

required_train_columns = {"problem", "problem_ko", "solution", "solution_ko"}
required_benchmark_columns = {"question", "question_ko", "final_answer"}
if not required_train_columns.issubset(train_dataset.column_names):
    raise ValueError(f"Missing training columns: {required_train_columns - set(train_dataset.column_names)}")
if not required_benchmark_columns.issubset(benchmark_dataset.column_names):
    raise ValueError(f"Missing benchmark columns: {required_benchmark_columns - set(benchmark_dataset.column_names)}")

print(f"Training candidates: {len(train_dataset):,}")
print(f"Benchmark rows: {len(benchmark_dataset):,}")

## 3. Exact normalized matching

English and Korean are checked independently. English is especially useful because translation wording may differ even when both datasets originated from the same problem.

In [ ]:
train_en = [normalize_math_text(text) for text in tqdm(train_dataset["problem"], desc="Normalizing train English")]
train_ko = [normalize_math_text(text) for text in tqdm(train_dataset["problem_ko"], desc="Normalizing train Korean")]
benchmark_en = [normalize_math_text(text) for text in benchmark_dataset["question"]]
benchmark_ko = [normalize_math_text(text) for text in benchmark_dataset["question_ko"]]

train_en_lookup = defaultdict(list)
train_ko_lookup = defaultdict(list)
for index, text in enumerate(train_en):
    if text:
        train_en_lookup[text].append(index)
for index, text in enumerate(train_ko):
    if text:
        train_ko_lookup[text].append(index)

exact_candidates_by_benchmark = []
for en_text, ko_text in zip(benchmark_en, benchmark_ko):
    candidates = set(train_en_lookup.get(en_text, []))
    candidates.update(train_ko_lookup.get(ko_text, []))
    exact_candidates_by_benchmark.append(candidates)

exact_training_indices = set().union(*exact_candidates_by_benchmark)
print(f"Normalized exact duplicate training rows: {len(exact_training_indices):,}")

## 4. Build or reload semantic embeddings

Embeddings are cached in Drive. The cache is accepted only when the model, row counts, and normalized-text fingerprints match the current datasets.

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

current_embedding_metadata = {
    "model_id": EMBEDDING_MODEL_ID,
    "train_rows": len(train_dataset),
    "benchmark_rows": len(benchmark_dataset),
    "train_fingerprint": text_fingerprint(train_dataset["problem"]),
    "benchmark_fingerprint": text_fingerprint(benchmark_dataset["question"]),
}

cached_metadata = None
if EMBEDDING_METADATA_PATH.exists():
    cached_metadata = json.loads(EMBEDDING_METADATA_PATH.read_text(encoding="utf-8"))

embedding_cache_valid = (
    not FORCE_REBUILD_EMBEDDINGS
    and TRAIN_EMBEDDINGS_PATH.exists()
    and BENCHMARK_EMBEDDINGS_PATH.exists()
    and cached_metadata == current_embedding_metadata
)

if embedding_cache_valid:
    print("Loading cached embeddings from Drive...")
    train_embeddings = np.load(TRAIN_EMBEDDINGS_PATH)
    benchmark_embeddings = np.load(BENCHMARK_EMBEDDINGS_PATH)
else:
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA is required, but PyTorch cannot see the GPU. Check the Colab runtime.")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        print(f"GPU: {properties.name}")
        print(f"GPU VRAM: {properties.total_memory / 1024**3:.1f} GiB")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    print(f"Encoding explicitly on {device} with batch size {EMBEDDING_BATCH_SIZE:,}.")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID, token=HF_TOKEN, device=device)
    if device.startswith("cuda") and USE_FP16_EMBEDDING_INFERENCE:
        embedding_model.half()
        print("Embedding model inference dtype: float16")
    embedding_model.eval()
    train_texts = list(train_dataset["problem"])
    benchmark_texts = list(benchmark_dataset["question"])
    train_embeddings = embedding_model.encode(
        train_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device,
    ).astype(np.float32)
    benchmark_embeddings = embedding_model.encode(
        benchmark_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device,
    ).astype(np.float32)
    np.save(TRAIN_EMBEDDINGS_PATH, train_embeddings)
    np.save(BENCHMARK_EMBEDDINGS_PATH, benchmark_embeddings)
    EMBEDDING_METADATA_PATH.write_text(
        json.dumps(current_embedding_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved embeddings to Drive.")

if train_embeddings.shape[0] != len(train_dataset):
    raise ValueError("Training embedding count does not match the dataset")
if benchmark_embeddings.shape[0] != len(benchmark_dataset):
    raise ValueError("Benchmark embedding count does not match the dataset")

print(f"Training embedding shape: {train_embeddings.shape}")
print(f"Benchmark embedding shape: {benchmark_embeddings.shape}")

## 5. Find near duplicates and create an audit

Each benchmark problem retrieves its nearest training candidates. A training row is removed when any of these conditions holds:

- normalized English or Korean text is identical,
- English or Korean fuzzy similarity reaches `FUZZY_REMOVE_THRESHOLD`, or
- semantic similarity reaches the deliberately strict `SEMANTIC_REMOVE_THRESHOLD`.

In [ ]:
import pandas as pd
from rapidfuzz import fuzz
from sentence_transformers import util

device = "cuda" if torch.cuda.is_available() else "cpu"
query_tensor = torch.from_numpy(benchmark_embeddings).to(device)
corpus_tensor = torch.from_numpy(train_embeddings).to(device)
semantic_hits = util.semantic_search(
    query_tensor,
    corpus_tensor,
    top_k=SEMANTIC_TOP_K,
    query_chunk_size=64,
    corpus_chunk_size=50000,
)
del query_tensor, corpus_tensor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

audit_rows = []
remove_indices = set()

for benchmark_index, hits in enumerate(semantic_hits):
    semantic_score_by_train = {hit["corpus_id"]: float(hit["score"]) for hit in hits}
    candidate_indices = set(semantic_score_by_train)
    candidate_indices.update(exact_candidates_by_benchmark[benchmark_index])

    for train_index in sorted(candidate_indices):
        exact_en = bool(benchmark_en[benchmark_index]) and train_en[train_index] == benchmark_en[benchmark_index]
        exact_ko = bool(benchmark_ko[benchmark_index]) and train_ko[train_index] == benchmark_ko[benchmark_index]
        fuzzy_en = float(fuzz.ratio(train_en[train_index], benchmark_en[benchmark_index]))
        fuzzy_ko = float(fuzz.ratio(train_ko[train_index], benchmark_ko[benchmark_index]))
        semantic_score = semantic_score_by_train.get(train_index)

        reasons = []
        if exact_en:
            reasons.append("exact_en")
        if exact_ko:
            reasons.append("exact_ko")
        if max(fuzzy_en, fuzzy_ko) >= FUZZY_REMOVE_THRESHOLD:
            reasons.append("fuzzy")
        if semantic_score is not None and semantic_score >= SEMANTIC_REMOVE_THRESHOLD:
            reasons.append("semantic")

        auto_remove = bool(reasons)
        if auto_remove:
            remove_indices.add(train_index)

        audit_rows.append({
            "benchmark_index": benchmark_index,
            "train_index": train_index,
            "auto_remove": auto_remove,
            "reasons": "|".join(reasons),
            "semantic_score": semantic_score,
            "fuzzy_english": fuzzy_en,
            "fuzzy_korean": fuzzy_ko,
            "benchmark_english": benchmark_dataset[benchmark_index]["question"],
            "training_english": train_dataset[train_index]["problem"],
            "benchmark_korean": benchmark_dataset[benchmark_index]["question_ko"],
            "training_korean": train_dataset[train_index]["problem_ko"],
        })

audit = pd.DataFrame(audit_rows).sort_values(
    ["auto_remove", "semantic_score", "fuzzy_english"],
    ascending=[False, False, False],
)
audit.to_csv(MATCH_AUDIT_PATH, index=False)

print(f"Training rows marked for removal: {len(remove_indices):,}")
print(f"Saved candidate audit: {MATCH_AUDIT_PATH}")
display(audit.head(20))

## 6. Save deduplicated data and summary to Drive

In [ ]:
keep_indices = [index for index in range(len(train_dataset)) if index not in remove_indices]
removed_indices_sorted = sorted(remove_indices)
deduplicated_dataset = train_dataset.select(keep_indices)
removed_dataset = train_dataset.select(removed_indices_sorted)

deduplicated_dataset.to_parquet(str(DEDUPLICATED_PATH))
removed_dataset.to_parquet(str(REMOVED_ROWS_PATH))

summary = {
    "training_dataset": TRAIN_DATASET_ID,
    "training_source_filter": TRAIN_SOURCE,
    "benchmark_dataset": BENCHMARK_DATASET_ID,
    "original_training_rows": len(train_dataset),
    "benchmark_rows": len(benchmark_dataset),
    "removed_training_rows": len(remove_indices),
    "remaining_training_rows": len(deduplicated_dataset),
    "exact_duplicate_training_rows": len(exact_training_indices),
    "fuzzy_remove_threshold": FUZZY_REMOVE_THRESHOLD,
    "semantic_remove_threshold": SEMANTIC_REMOVE_THRESHOLD,
    "embedding_model": EMBEDDING_MODEL_ID,
    "deduplicated_parquet": str(DEDUPLICATED_PATH),
    "removed_rows_parquet": str(REMOVED_ROWS_PATH),
    "match_audit_csv": str(MATCH_AUDIT_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nAll outputs are safely persisted in Google Drive.")

## 7. Download notebook outputs

The compact audit and summary download automatically. Set `DOWNLOAD_DEDUPLICATED_PARQUET = True` in the controls if you also want the large training Parquet downloaded to your computer; it is always saved in Drive regardless.

In [ ]:
from google.colab import files

files.download(str(SUMMARY_PATH))
files.download(str(MATCH_AUDIT_PATH))

if DOWNLOAD_DEDUPLICATED_PARQUET:
    files.download(str(DEDUPLICATED_PATH))
else:
    print(f"Large deduplicated dataset remains available at: {DEDUPLICATED_PATH}")